# **🏋🏽 Wearable Biomechanics - Dataset Export & Reports**

### 🔧 Colab Setup & Library Init

In [ ]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & configure renderers
try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "cleannjerk_dataset" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.export import export_dataset, load_and_deduplicate_entries, build_id_mapping_df

source_base_dir = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}
output_dir = "/content/drive/MyDrive/Academic/Research/Wearable Sensing/Wearable Biomechanics/Clean&JerkDataset" # @param {type:"string"}
mapping_csv_path = os.path.join(repo_root, "docs", "export_id_mapping.csv")

### 🔍 Preview Chronological ID Mapping

In [ ]:
# @title 🔍 Preview Chronological ID Mapping
records, sub_map, _ = load_and_deduplicate_entries(
    os.path.join(source_base_dir, "index"),
    existing_mapping_csv=mapping_csv_path if os.path.exists(mapping_csv_path) else None,
)
mdf = build_id_mapping_df(records)
print(f"Total unique Sets: {len(mdf)} | Total Participants: {len(sub_map)}")
display_table(mdf, clean_columns=False)

### 🚀 Run Dataset Export (Batch or Incremental)

In [ ]:
# @title 🚀 Run Dataset Export { display-mode: "form" }
only_new = False # @param {type:"boolean"}
copy_videos = True # @param {type:"boolean"}

exported = export_dataset(
    source_base_dir=source_base_dir,
    output_dir=output_dir,
    mapping_csv_path=mapping_csv_path,
    copy_videos=copy_videos,
    only_new=only_new,
)
print(f"\n✅ Export complete to: {output_dir}")
print(f"📝 ID mapping saved to: {mapping_csv_path}")

### ✅ Inspect Exported Summary Tables

In [ ]:
# @title ✅ Inspect Exported Summary Tables
display_table(exported["participants"], clean_columns=False)
display_table(exported["sets"], clean_columns=False)
display_table(exported["devices"], clean_columns=False)
display_table(exported["phase_markers"].head(50), clean_columns=False)